# RIN - Experiment 6c: diagnostic benchmark for denominator families

This notebook fixes the source event explicitly, removes query-gated diffusion after the source is chosen, and compares several local denominator families on a set of small diagnostic motifs.

The goal is not a full retrieval benchmark. The goal is a cleaner mechanism test for local retention laws.


In [1]:
import sys
from pathlib import Path

def find_repo_root(marker="requirements.txt"):
    path = Path.cwd()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"could not find repo root (looked for {marker})")

REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("repo root:", REPO_ROOT)


repo root: C:\Users\yaros\BlackSabbitch_repo\RIN


In [ ]:
%load_ext autoreload
%autoreload 2


## Graph definition

The benchmark graph contains four diagnostic regions: weather, kit, travel, and metabolic reasoning.


In [2]:
from collections import defaultdict
from itertools import product

import numpy as np

from embryonic.numberbatch import Numberbatch
from embryonic.query import longest_match_terms
from memory.edge import Edge
from memory.graph import MemoryGraph
from memory.node import Node

NUMBERBATCH_FILE = REPO_ROOT / "source" / "numberbatch-en-19.08.txt.gz"

NODE_SPECS = [
    ("umbrella", "/c/en/umbrella", "concept", 0.7, 1.0, "weather"),
    ("outdoor gear", None, "concept", 0.7, 2.5, "weather"),
    ("rain coat", "/c/en/raincoat", "concept", 0.7, 1.2, "weather"),
    ("boots", "/c/en/boot", "concept", 0.7, 1.2, "weather"),
    ("gloves", "/c/en/glove", "concept", 0.7, 1.2, "weather"),
    ("storm signal", None, "episode", 0.8, 1.5, "weather"),
    ("camp overview", None, "episode", 0.8, 1.5, "weather"),
    ("organizer pouch", None, "concept", 0.7, 1.0, "kit"),
    ("desk drawer", None, "concept", 0.7, 1.2, "kit"),
    ("cable", "/c/en/cable", "concept", 0.7, 1.0, "kit"),
    ("pen", "/c/en/pen", "concept", 0.7, 1.0, "kit"),
    ("coin", "/c/en/coin", "concept", 0.7, 1.0, "kit"),
    ("ticket", "/c/en/ticket", "concept", 0.7, 1.0, "kit"),
    ("office note", None, "concept", 0.7, 1.0, "kit"),
    ("audit ping", None, "episode", 0.8, 1.5, "kit"),
    ("passport", "/c/en/passport", "concept", 0.7, 1.0, "travel"),
    ("identity", "/c/en/identity", "concept", 0.7, 2.5, "travel"),
    ("border ping", None, "episode", 0.8, 1.5, "travel"),
    ("document", "/c/en/document", "concept", 0.7, 1.2, "travel"),
    ("travel", "/c/en/travel", "concept", 0.7, 2.0, "travel"),
    ("coffee", "/c/en/coffee", "concept", 0.7, 1.5, "health"),
    ("caffeine", "/c/en/caffeine", "concept", 0.7, 1.0, "health"),
    ("beverage", "/c/en/beverage", "concept", 0.7, 2.5, "health"),
    ("energy", "/c/en/energy", "concept", 0.7, 1.5, "health"),
    ("water", "/c/en/water", "concept", 0.7, 1.2, "health"),
    ("metabolic puzzle", None, "episode", 0.8, 1.5, "health"),
]

EDGE_SPECS = [
    ("storm signal", "umbrella", 1.15),
    ("storm signal", "outdoor gear", 1.25),
    ("umbrella", "rain coat", 0.2),
    ("umbrella", "boots", 0.2),
    ("umbrella", "gloves", 0.2),
    ("outdoor gear", "rain coat", 1.0),
    ("outdoor gear", "boots", 1.0),
    ("outdoor gear", "gloves", 1.0),
    ("camp overview", "outdoor gear", 1.7),
    ("camp overview", "umbrella", 0.8),
    ("camp overview", "rain coat", 0.8),
    ("audit ping", "organizer pouch", 1.25),
    ("audit ping", "desk drawer", 1.15),
    ("organizer pouch", "cable", 0.5),
    ("organizer pouch", "pen", 0.5),
    ("organizer pouch", "coin", 0.5),
    ("organizer pouch", "ticket", 0.5),
    ("desk drawer", "office note", 2.0),
    ("border ping", "passport", 1.0),
    ("border ping", "identity", 1.0),
    ("passport", "document", 1.0),
    ("passport", "travel", 1.0),
    ("identity", "document", 1.0),
    ("identity", "travel", 1.0),
    ("metabolic puzzle", "caffeine", 1.0),
    ("metabolic puzzle", "beverage", 1.15),
    ("coffee", "caffeine", 1.5),
    ("coffee", "beverage", 1.0),
    ("coffee", "energy", 1.2),
    ("beverage", "water", 1.2),
    ("beverage", "energy", 0.6),
]

QUERY_SPECS = [
    ("Q1", "what did the storm signal indicate", "storm signal", ["umbrella"], ["outdoor gear"]),
    ("Q2", "what did the audit ping indicate", "audit ping", ["organizer pouch"], ["desk drawer"]),
    ("Q3", "what did the border ping indicate", "border ping", ["passport"], ["identity"]),
    ("Q4", "tell me about the camp overview", "camp overview", ["outdoor gear"], ["umbrella", "rain coat"]),
    ("Q5", "what explained the metabolic puzzle", "metabolic puzzle", ["caffeine"], ["beverage"]),
]

FORMULA_GRIDS = {
    "R0": [{}],
    "R1": [dict(beta=b, gamma=g) for b, g in product([0.05, 0.1, 0.25, 0.5, 0.75, 1.0], [0.5, 0.75, 1.0, 1.25, 1.5])],
    "R2": [dict(beta=b, gamma=g) for b, g in product([0.05, 0.1, 0.25, 0.5, 0.75, 1.0], [0.5, 0.75, 1.0, 1.25, 1.5])],
    "R3": [dict(alpha=a, beta=b, gamma=g) for a, b, g in product([1.0, 1.5, 2.0], [0.05, 0.1, 0.25, 0.5, 0.75], [0.5, 0.75, 1.0, 1.25])],
    "R4": [dict(beta=b, gamma=g) for b, g in product([0.05, 0.1, 0.25, 0.5, 0.75, 1.0], [0.5, 0.75, 1.0, 1.25, 1.5])],
    "R5": [dict(alpha=a, beta=b, gamma=g) for a, b, g in product([1.0, 1.5, 2.0], [0.05, 0.1, 0.25, 0.5, 0.75], [0.5, 0.75, 1.0, 1.25])],
}

POSITIONS = {
    "storm signal": (-5.2, 1.4),
    "umbrella": (-4.2, 1.3),
    "outdoor gear": (-3.0, 1.8),
    "rain coat": (-2.3, 1.0),
    "boots": (-2.8, 0.0),
    "gloves": (-1.9, 0.2),
    "camp overview": (-4.2, 2.8),
    "audit ping": (0.0, 4.0),
    "organizer pouch": (-0.8, 2.8),
    "desk drawer": (1.0, 2.8),
    "cable": (-1.8, 1.8),
    "pen": (-0.9, 1.5),
    "coin": (-0.1, 1.5),
    "ticket": (0.8, 1.7),
    "office note": (1.9, 1.8),
    "border ping": (4.0, 3.8),
    "passport": (3.3, 2.8),
    "identity": (4.7, 2.8),
    "document": (3.6, 1.8),
    "travel": (4.5, 1.7),
    "metabolic puzzle": (0.0, -2.0),
    "coffee": (-1.2, -3.2),
    "caffeine": (-2.2, -4.0),
    "beverage": (1.2, -3.8),
    "energy": (-0.2, -4.8),
    "water": (2.3, -4.7),
}

CLUSTER_COLORS = {
    "weather": "#d6b06a",
    "kit": "#7db7d5",
    "travel": "#9fc46b",
    "health": "#d88478",
}

NODE_CAPACITY = {name: capacity for name, _uri, _kind, _conf, capacity, _cluster in NODE_SPECS}
NODE_CLUSTER = {name: cluster for name, _uri, _kind, _conf, _cap, cluster in NODE_SPECS}

numberbatch = Numberbatch()
numberbatch.load(str(NUMBERBATCH_FILE))
print(f"loaded {len(numberbatch.terms):,} terms")


def phrase_embedding(text):
    uris = longest_match_terms(text, numberbatch)
    vecs = [numberbatch.vector_for(uri) for uri in uris if numberbatch.vector_for(uri) is not None]
    if not vecs:
        return None
    return np.mean(np.stack(vecs).astype(np.float32), axis=0)


def build_graph():
    G = MemoryGraph()
    content_to_id = {}
    for i, (content, uri, kind, conf, _capacity, _cluster) in enumerate(NODE_SPECS):
        vector = numberbatch.vector_for(uri) if uri is not None else None
        if vector is None:
            vector = phrase_embedding(content)
        G.add_node(Node(id=i, content=content, embedding=vector, node_kind=kind, confidence=conf, provenance=[uri or content]))
        content_to_id[content] = i
    for src, dst, weight in EDGE_SPECS:
        G.add_edge(Edge(src=content_to_id[src], dst=content_to_id[dst], type="RelatedTo", weight=weight))
        G.add_edge(Edge(src=content_to_id[dst], dst=content_to_id[src], type="RelatedTo", weight=weight))
    return G, content_to_id


G, content_to_id = build_graph()
print(f"nodes: {len(G.nodes)}")
print(f"undirected motifs: {len(EDGE_SPECS)}")


loaded 516,782 terms
nodes: 26
undirected motifs: 31


## Optional graph picture

If the plotting stack is available, draw the hand-authored diagnostic world.


In [3]:
try:
    import matplotlib.pyplot as plt
    import networkx as nx
except ModuleNotFoundError as exc:
    print("graph plot skipped:", exc)
else:
    H = nx.Graph()
    for name, _uri, _kind, _conf, capacity, cluster in NODE_SPECS:
        H.add_node(name, capacity=capacity, cluster=cluster)
    for src, dst, weight in EDGE_SPECS:
        H.add_edge(src, dst, weight=weight)

    colors = [CLUSTER_COLORS[H.nodes[name]["cluster"]] for name in H.nodes]
    sizes = [550 + 180 * H.nodes[name]["capacity"] for name in H.nodes]
    widths = [0.7 + 0.9 * data["weight"] for *_edge, data in H.edges(data=True)]

    plt.figure(figsize=(11, 8))
    nx.draw_networkx_edges(H, POSITIONS, width=widths, alpha=0.45, edge_color="#666666")
    nx.draw_networkx_nodes(H, POSITIONS, node_color=colors, node_size=sizes, linewidths=1.0, edgecolors="#222222")
    nx.draw_networkx_labels(H, POSITIONS, font_size=8)
    plt.title("Experiment 6c diagnostic graph")
    plt.axis("off")
    plt.show()


graph plot skipped: No module named 'matplotlib'


## Query transparency

Each query fixes its source event explicitly. The query text remains only for readability and term matching inspection.


In [4]:
for qid, query_text, source_event, correct_nodes, distractor_nodes in QUERY_SPECS:
    print(qid, query_text)
    print("  matched terms:", longest_match_terms(query_text, numberbatch))
    print("  source event:", source_event)
    print("  correct:", correct_nodes)
    print("  distractors:", distractor_nodes)
    print()


Q1 what did the storm signal indicate
  matched terms: ['/c/en/storm_signal', '/c/en/indicate']
  source event: storm signal
  correct: ['umbrella']
  distractors: ['outdoor gear']

Q2 what did the audit ping indicate
  matched terms: ['/c/en/audit', '/c/en/ping', '/c/en/indicate']
  source event: audit ping
  correct: ['organizer pouch']
  distractors: ['desk drawer']

Q3 what did the border ping indicate
  matched terms: ['/c/en/border', '/c/en/ping', '/c/en/indicate']
  source event: border ping
  correct: ['passport']
  distractors: ['identity']

Q4 tell me about the camp overview
  matched terms: ['/c/en/camp', '/c/en/overview']
  source event: camp overview
  correct: ['outdoor gear']
  distractors: ['umbrella', 'rain coat']

Q5 what explained the metabolic puzzle
  matched terms: ['/c/en/explained', '/c/en/metabolic', '/c/en/puzzle']
  source event: metabolic puzzle
  correct: ['caffeine']
  distractors: ['beverage']



## Retention-law helpers

Diffusion starts from the explicit source node and then uses only weighted propagation plus a local denominator family.


In [5]:
def out_degree(G, node_id):
    return len(G.out_edges(node_id))


def weighted_out_degree(G, node_id):
    return sum(edge.weight for edge in G.out_edges(node_id))


def denominator_builder(formula_name, params):
    alpha = params.get("alpha", 1.0)
    beta = params.get("beta", 1.0)
    gamma = params.get("gamma", 1.0)

    def denominator(node_id):
        k_out = max(1, out_degree(G, node_id))
        s_out = max(1e-6, weighted_out_degree(G, node_id))
        capacity = NODE_CAPACITY[G.nodes[node_id].content]
        if formula_name == "R0":
            return 1.0
        if formula_name == "R1":
            return 1.0 + beta * (k_out ** gamma)
        if formula_name == "R2":
            return 1.0 + beta * (s_out ** gamma)
        if formula_name == "R3":
            return alpha + beta * (s_out ** gamma)
        if formula_name == "R4":
            return capacity + beta * (s_out ** gamma)
        if formula_name == "R5":
            return alpha + capacity + beta * (s_out ** gamma)
        raise KeyError(formula_name)

    return denominator


def diffuse_with_retention(G, source_node_id, denominator, initial_activation=100.0, decay=0.8, floor_fraction=0.01, max_steps=20):
    floor = floor_fraction * initial_activation
    current = {source_node_id: initial_activation}
    peak = dict(current)

    for _step in range(max_steps):
        active = {node_id: amount for node_id, amount in current.items() if amount >= floor}
        if not active:
            break

        incoming = defaultdict(float)
        for node_id, amount in active.items():
            out_edges = G.out_edges(node_id)
            total_weight = sum(edge.weight for edge in out_edges)
            if total_weight <= 0:
                continue
            for edge in out_edges:
                incoming[edge.dst] += decay * amount * (edge.weight / total_weight)

        current = {dst: amount / max(1e-6, denominator(dst)) for dst, amount in incoming.items()}
        for node_id, amount in current.items():
            if amount > peak.get(node_id, 0.0):
                peak[node_id] = amount

    return peak


def evaluate_formula(formula_name, params, query_specs=None):
    active_queries = QUERY_SPECS if query_specs is None else query_specs
    solved = 0
    total_margin = 0.0
    rows = []

    for qid, query_text, source_event, correct_nodes, distractor_nodes in active_queries:
        source_id = content_to_id[source_event]
        peak = diffuse_with_retention(G, source_id, denominator_builder(formula_name, params))
        best_correct = max(peak.get(content_to_id[name], 0.0) for name in correct_nodes)
        best_distractor = max(peak.get(content_to_id[name], 0.0) for name in distractor_nodes)
        success = best_correct > best_distractor
        solved += int(success)
        total_margin += best_correct - best_distractor
        rows.append({
            "id": qid,
            "query": query_text,
            "source_event": source_event,
            "best_correct": best_correct,
            "best_distractor": best_distractor,
            "success": success,
        })

    return solved, total_margin, rows


def select_best_parameters(formula_name, query_specs=None):
    best = None
    for params in FORMULA_GRIDS[formula_name]:
        solved, total_margin, rows = evaluate_formula(formula_name, params, query_specs=query_specs)
        candidate = (solved, total_margin, params, rows)
        if best is None or candidate[:2] > best[:2]:
            best = candidate
    return best


def print_rows(rows):
    for row in rows:
        print(
            f"{row['id']}  correct={row['best_correct']:.2f}  distractor={row['best_distractor']:.2f}  "
            f"success={row['success']}  source={row['source_event']}"
        )


## Parameter landscapes

Print the best few settings in each searched family before plotting.


In [6]:
def parameter_names_for_formula(formula_name):
    names = []
    for key in ["alpha", "beta", "gamma"]:
        if any(key in params for params in FORMULA_GRIDS[formula_name]):
            names.append(key)
    return names


parameter_landscapes = {}
for formula_name, grid in FORMULA_GRIDS.items():
    records = []
    for params in grid:
        solved, total_margin, _rows = evaluate_formula(formula_name, params)
        record = {
            "formula": formula_name,
            "solved": solved,
            "margin": total_margin,
            "params": dict(params),
        }
        for key in parameter_names_for_formula(formula_name):
            record[key] = params.get(key)
        records.append(record)
    parameter_landscapes[formula_name] = records

for formula_name, records in parameter_landscapes.items():
    print(formula_name, "parameter count:", len(records))
    best_records = sorted(records, key=lambda r: (r["solved"], r["margin"]), reverse=True)[:5]
    for record in best_records:
        print("  ", record["params"], "->", f"{record['solved']}/{len(QUERY_SPECS)}", f"margin={record['margin']:.2f}")
    print()


R0 parameter count: 1
   {} -> 2/5 margin=16.24

R1 parameter count: 30
   {'beta': 0.05, 'gamma': 0.5} -> 2/5 margin=13.75
   {'beta': 0.05, 'gamma': 0.75} -> 2/5 margin=12.22
   {'beta': 0.1, 'gamma': 0.5} -> 2/5 margin=11.83
   {'beta': 0.1, 'gamma': 1.0} -> 2/5 margin=6.61
   {'beta': 0.25, 'gamma': 0.75} -> 2/5 margin=5.12

R2 parameter count: 30
   {'beta': 0.05, 'gamma': 1.5} -> 4/5 margin=17.20
   {'beta': 0.1, 'gamma': 1.5} -> 4/5 margin=16.32
   {'beta': 0.1, 'gamma': 1.25} -> 4/5 margin=15.92
   {'beta': 0.25, 'gamma': 1.0} -> 4/5 margin=13.21
   {'beta': 0.25, 'gamma': 1.25} -> 4/5 margin=13.10

R3 parameter count: 60
   {'alpha': 1.0, 'beta': 0.1, 'gamma': 1.25} -> 4/5 margin=15.92
   {'alpha': 1.0, 'beta': 0.25, 'gamma': 1.0} -> 4/5 margin=13.21
   {'alpha': 1.0, 'beta': 0.25, 'gamma': 1.25} -> 4/5 margin=13.10
   {'alpha': 1.0, 'beta': 0.5, 'gamma': 0.75} -> 4/5 margin=10.51
   {'alpha': 1.0, 'beta': 0.5, 'gamma': 1.0} -> 4/5 margin=10.18

R4 parameter count: 30
   {'bet

## Optional parameter plots

Plot the searched parameter grids. When a family has three free parameters, the third appears as color.


In [7]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    print("parameter plots skipped:", exc)
else:
    def scatter_sizes(records):
        margins = np.array([record["margin"] for record in records], dtype=np.float32)
        if len(margins) == 0 or float(margins.max() - margins.min()) < 1e-9:
            return np.full(len(records), 120.0)
        scaled = (margins - margins.min()) / (margins.max() - margins.min())
        return 80.0 + 280.0 * scaled


    def annotate_top_points(ax, records, xname, yname, top_n=5):
        top_records = sorted(records, key=lambda r: (r["solved"], r["margin"]), reverse=True)[:top_n]
        for record in top_records:
            label = f"{record['solved']}/{len(QUERY_SPECS)}\n{record['margin']:.1f}"
            ax.annotate(label, (record[xname], record[yname]), textcoords="offset points", xytext=(5, 5), fontsize=7)


    def plot_formula_landscape(formula_name):
        records = parameter_landscapes[formula_name]
        names = parameter_names_for_formula(formula_name)
        if not names:
            record = records[0]
            print(formula_name, "has no free parameters:", f"{record['solved']}/{len(QUERY_SPECS)}", f"margin={record['margin']:.2f}")
            return
        if len(names) == 1:
            xname = names[0]
            xs = [record[xname] for record in records]
            ys = [record["margin"] for record in records]
            colors = [record["solved"] for record in records]
            sizes = scatter_sizes(records)
            fig, ax = plt.subplots(figsize=(6, 4))
            scatter = ax.scatter(xs, ys, c=colors, s=sizes, cmap="coolwarm", alpha=0.85, edgecolors="#333333")
            ax.set_xlabel(xname)
            ax.set_ylabel("margin")
            ax.set_title(f"{formula_name}: 1D landscape")
            for record in sorted(records, key=lambda r: (r['solved'], r['margin']), reverse=True)[:5]:
                ax.annotate(f"{record['solved']}/{len(QUERY_SPECS)}", (record[xname], record['margin']), textcoords="offset points", xytext=(5, 5), fontsize=7)
            cbar = plt.colorbar(scatter, ax=ax)
            cbar.set_label("queries solved")
            plt.show()
            return

        xname, yname = names[0], names[1]
        xs = [record[xname] for record in records]
        ys = [record[yname] for record in records]
        sizes = scatter_sizes(records)
        if len(names) >= 3:
            colors = [record[names[2]] for record in records]
            color_label = names[2]
        else:
            colors = [record["margin"] for record in records]
            color_label = "margin"

        fig, ax = plt.subplots(figsize=(6.5, 5))
        scatter = ax.scatter(xs, ys, c=colors, s=sizes, cmap="coolwarm", alpha=0.85, edgecolors="#333333")
        annotate_top_points(ax, records, xname, yname, top_n=5)
        ax.set_xlabel(xname)
        ax.set_ylabel(yname)
        ax.set_title(f"{formula_name}: parameter landscape")
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label(color_label)
        plt.show()


    for formula_name in FORMULA_GRIDS:
        plot_formula_landscape(formula_name)


parameter plots skipped: No module named 'matplotlib'


## In-sample best parameters per family

Choose the best grid setting on the full five-query suite.


In [8]:
global_bests = {}
for formula_name in FORMULA_GRIDS:
    best = select_best_parameters(formula_name)
    global_bests[formula_name] = best
    solved, total_margin, params, rows = best
    print(formula_name, solved, round(total_margin, 2), params)
    print_rows(rows)
    print()


R0 2 16.24 {}
Q1  correct=38.33  distractor=41.67  success=False  source=storm signal
Q2  correct=41.67  distractor=38.33  success=True  source=audit ping
Q3  correct=40.00  distractor=40.00  success=False  source=border ping
Q4  correct=41.21  distractor=19.39  success=True  source=camp overview
Q5  correct=37.21  distractor=42.79  success=False  source=metabolic puzzle

R1 2 13.75 {'beta': 0.05, 'gamma': 0.5}
Q1  correct=34.48  distractor=37.48  success=False  source=storm signal
Q2  correct=37.48  distractor=35.80  success=True  source=audit ping
Q3  correct=36.81  distractor=36.81  success=False  source=border ping
Q4  correct=37.07  distractor=17.85  success=True  source=camp overview
Q5  correct=34.75  distractor=38.90  success=False  source=metabolic puzzle

R2 4 17.2 {'beta': 0.05, 'gamma': 1.5}
Q1  correct=31.85  distractor=24.15  success=True  source=storm signal
Q2  correct=32.23  distractor=29.96  success=True  source=audit ping
Q3  correct=31.75  distractor=31.75  success=

## Leave-one-query-out validation

For each held-out query, fit parameters on the remaining four and evaluate on the fifth.


In [9]:
loo_summary = {}
for formula_name in FORMULA_GRIDS:
    held_out_solved = 0
    held_out_margin = 0.0
    chosen_params = []
    for i, held_out in enumerate(QUERY_SPECS):
        train_queries = QUERY_SPECS[:i] + QUERY_SPECS[i + 1 :]
        best_train = select_best_parameters(formula_name, query_specs=train_queries)
        chosen_params.append(best_train[2])
        solved, total_margin, _rows = evaluate_formula(formula_name, best_train[2], query_specs=[held_out])
        held_out_solved += solved
        held_out_margin += total_margin
    loo_summary[formula_name] = {
        "solved": held_out_solved,
        "margin": held_out_margin,
        "params": chosen_params,
    }
    print(formula_name, held_out_solved, round(held_out_margin, 2))


R0 2 16.24
R1 1 -13.75
R2 2 4.86
R3 3 6.98
R4 4 29.51
R5 5 26.19


## Winning family details

Compare the best in-sample family with the best held-out family.


In [10]:
winning_formula_in_sample = max(global_bests.items(), key=lambda item: (item[1][0], item[1][1]))[0]
winning_formula_held_out = max(loo_summary.items(), key=lambda item: (item[1]["solved"], item[1]["margin"]))[0]

print("winning formula in-sample:", winning_formula_in_sample)
print("best in-sample params:", global_bests[winning_formula_in_sample][2])
print()
print_rows(global_bests[winning_formula_in_sample][3])
print()
print("winning formula by held-out score:", winning_formula_held_out)
print("best global params for held-out winner:", global_bests[winning_formula_held_out][2])
print()
print_rows(global_bests[winning_formula_held_out][3])


winning formula in-sample: R4
best in-sample params: {'beta': 0.5, 'gamma': 0.5}

Q1  correct=21.31  distractor=11.20  success=True  source=storm signal
Q2  correct=21.91  distractor=18.36  success=True  source=audit ping
Q3  correct=21.44  distractor=11.88  success=True  source=border ping
Q4  correct=11.08  distractor=10.78  success=True  source=camp overview
Q5  correct=20.78  distractor=12.25  success=True  source=metabolic puzzle

winning formula by held-out score: R5
best global params for held-out winner: {'alpha': 1.0, 'beta': 0.05, 'gamma': 0.5}

Q1  correct=18.43  distractor=11.50  success=True  source=storm signal
Q2  correct=19.93  distractor=16.75  success=True  source=audit ping
Q3  correct=19.17  distractor=11.15  success=True  source=border ping
Q4  correct=11.38  distractor=9.32  success=True  source=camp overview
Q5  correct=17.90  distractor=11.89  success=True  source=metabolic puzzle
